In [ ]:
import sys
sys.path.append("../")
from d2l import torch as d2l
%matplotlib inline
import random
import torch

In [ ]:
def synthetic_data(w, b, num_examples):
    """Generate y = Xw + b + noise."""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))


#生成小批量随机数据迭代器
def data_iter(batch_size, features, labels):

    #总样本数
    num_examples = len(features)

    #随机打乱样本索引
    indices = list(range(num_examples))
    random.shuffle(indices)

    #每次取batch_size个样本
    for i in range(0, num_examples, batch_size):
        #切片操作
        j = torch.tensor(indices[i:min(i + batch_size, num_examples)])
        yield features.index_select(0, j), labels.index_select(0, j)


def linreg(X, w, b):  #@save
    """线性回归模型"""
    return torch.matmul(X, w) + b

def squared_loss(y_hat, y):  #@save
    """均方损失"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

def sgd(params, lr, batch_size):  #@save
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)


w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)


lr = 0.03
num_epochs = 5
net = linreg
loss = squared_loss
batch_size = 10

# 代码1:
for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        y_hat = net(X, w, b)
        l = loss(y_hat, y)  # X和y的小批量损失
        # 因为l形状是(batch_size,1)，而不是一个标量。l中的所有元素被加到一起，
        # 并以此计算关于[w,b]的梯度
        l.sum().backward()
        sgd([w, b], lr, batch_size)  # 使用参数的梯度更新参数
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

# 代码2
for i in range(epoch):
    for x, y in data_iter(batch_size, features, labels):
        y_hat = torch.matmul(x, w) + b
        loss = (y_hat - y) ** 2 / 2
        loss.sum().backward()

    #更新参数 小批量梯度下降
    # sgd(p, lr, batch_size)
        with torch.no_grad():
            w -= lr * w.grad / batch_size
            w.grad.zero_()
            b -= lr * b.grad / batch_size
            b.grad.zero_()
        
        
    with torch.no_grad():
        y_hat = torch.matmul(features , w) + b
        tl = (y_hat - labels) ** 2 / 2

    print(f'epoch {epoch + 1}, loss {float(tl.mean()):f}')


epoch 5, loss 0.040159
epoch 5, loss 0.000152
epoch 5, loss 0.000051
epoch 5, loss 0.000051


In [ ]:
def synthetic_data(max_num, w, b):
    X = torch.normal(0, 0.1, (max_num, len(w)))
    Y = torch.matmul(X,w) + b
    Y += torch.normal(0, 0.1, Y.shape)
    return X, Y

def data_iter(batch_size, features, labels):
    num = len(features)
    index = list(range(num))
    random.shuffle(index)
    
    for i in range(0, num, batch_size):
        j = torch.tensor(index[i:min(i + batch_size, num)])
        yield features.index_select(0, j), labels.index_select(0, j)

def linear_recession(X, w, b):
    return torch.matmul(X, w) + b

def square_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

def sgd(batch_size, lr, params):
    with torch.no_grad():
        for para in params:
            para -= lr * para.grad / batch_size
            para.grad.zero_()

true_w = torch.tensor([2.3, -3.2])
true_b = torch.tensor([3.2])

features, labels = synthetic_data(1000, true_w, true_b)

#hyper parameters
epoch = 3
batch_size = 10
lr = 0.03
net = linear_recession
loss = square_loss

#init
w = torch.tensor(torch.normal(0, 0.1, (2,)), requires_grad = True)
b = torch.tensor([1.0], requires_grad = True)

for i in range(epoch):
    for x, y in data_iter(batch_size, features, labels):
        l = loss(net(x, w, b),  y)
        l.sum().backward()

        sgd(batch_size, lr, [w,b])

    l = loss(net(features, w, b), labels)
    print(f"epoch{i + 1}, loss{float(l.mean()):f}")
